# KV-cache pruning experiments on Kaggle (2x T4)

**One-time setup:** phone-verify your Kaggle account. In this notebook's right panel set **Accelerator = GPU T4 x2** and **Internet = On**.

**How to run a stage unattended (recommended):**
1. Set `STAGE` in the next cell.
2. Click **Save Version → Save & Run All (Commit)**. It runs in the background for up to 12 hours; you can close the browser.
3. When it finishes, open the version's **Output** tab and download `results.zip` (also commit the `results/` files to GitHub).

**Next stage:** click **Add Input → Your Work → this notebook's previous version**, change `STAGE`, and Save Version again. Earlier results are restored automatically and finished rows are skipped.

| STAGE | What runs | Approx. time on 2x T4 |
| --- | --- | --- |
| `main_20` | static profile calibration + main experiment at 20% budget | 3-4 h |
| `main_rest` | main experiment at 10%, 30%, 50% | 7-9 h |
| `passkey_ablation` | passkey heatmaps (GPU 0) + tau/floor/random ablations (GPU 1) | 4-6 h |
| `window_profile` | window 16 / 64 ablations + memory/latency profiling | 3-4 h |
| `llama` | Llama-3.2-3B generalisation (needs HF_TOKEN secret) | 3-4 h |
| `figures` | only regenerate figures from restored results | 2 min |

In [ ]:
STAGE = "main_20"     # main_20 | main_rest | passkey_ablation | window_profile | llama | figures
REPO = "https://github.com/khushpatel3016/kv-cache-entropy-pruning.git"

In [ ]:
import os, subprocess, glob, shutil, time, sys
os.chdir("/kaggle/working")
if not os.path.exists("kvp"):
    subprocess.run(["git", "clone", "-q", REPO, "kvp"], check=True)
os.chdir("/kaggle/working/kvp")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==5.20.1", "datasets==3.6.0",
                "rapidfuzz", "tabulate", "pyyaml", "-e", "."], check=True)
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"])
os.makedirs("results", exist_ok=True); os.makedirs("profiles", exist_ok=True)

# Restore results and profiles from any attached input (earlier versions of this notebook or a dataset).
for f in glob.glob("/kaggle/input/**/results/*.jsonl", recursive=True):
    dst = os.path.join("results", os.path.basename(f))
    if not os.path.exists(dst) or os.path.getsize(f) > os.path.getsize(dst):
        shutil.copy(f, dst); print("restored", f)
for f in glob.glob("/kaggle/input/**/profiles/*.json", recursive=True):
    shutil.copy(f, "profiles/"); print("restored", f)

In [ ]:
# Hugging Face token (only needed for Llama-3.2). Add it under Add-ons > Secrets as HF_TOKEN.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF token loaded")
except Exception as e:
    print("no HF token (fine for Qwen):", type(e).__name__)

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)

In [ ]:
ENV = {**os.environ, "PYTHONPATH": "src", "TOKENIZERS_PARALLELISM": "false"}
SPLIT = {0: ["qasper", "hotpotqa", "gov_report"], 1: ["trec", "passage_retrieval_en", "lcc"]}

def sh(cmd, gpu=0, log=None):
    env = {**ENV, "CUDA_VISIBLE_DEVICES": str(gpu)}
    out = open(log, "a") if log else None
    return subprocess.Popen(cmd, env=env, stdout=out, stderr=subprocess.STDOUT if out else None)

def wait(procs, logs):
    while any(p.poll() is None for p in procs):
        time.sleep(300)
        for lg in logs:
            try: print(lg, "|", open(lg).read().strip().splitlines()[-1][:150])
            except Exception: pass
    codes = [p.returncode for p in procs]
    print("exit codes", codes)
    for lg in logs:
        print("----", lg); print("".join(open(lg).readlines()[-15:]))
    if any(codes): raise RuntimeError(f"a job failed: {codes}; see logs above")

def run_cfg(cfg, extra=(), gpu=0, tag=None):
    tag = tag or os.path.basename(cfg).replace(".yaml", "")
    return sh([sys.executable, "-m", "kvprune.run", "--config", cfg, *extra], gpu, f"log_{tag}.txt"), f"log_{tag}.txt"

def main_split(cfg, budgets, out_prefix):
    jobs = [run_cfg(cfg, ["--budgets", *budgets, "--tasks", *tasks, "--output", f"results/{out_prefix}_gpu{g}.jsonl"],
                    gpu=g, tag=f"{out_prefix}_gpu{g}") for g, tasks in SPLIT.items()]
    wait([j[0] for j in jobs], [j[1] for j in jobs])

def calibrate(model, out):
    if os.path.exists(out): print("profile exists:", out); return
    p = sh([sys.executable, "scripts/calibrate_profile.py", "--model", model, "--out", out], 0, "log_calibrate.txt")
    wait([p], ["log_calibrate.txt"])

In [ ]:
t0 = time.time()
if STAGE == "main_20":
    calibrate("Qwen/Qwen2.5-3B-Instruct", "profiles/qwen2.5-3b-instruct.json")
    main_split("configs/main_qwen3b.yaml", ["0.2"], "main_qwen3b")
elif STAGE == "main_rest":
    main_split("configs/main_qwen3b.yaml", ["0.1", "0.3", "0.5"], "main_qwen3b")
elif STAGE == "passkey_ablation":
    a = run_cfg("configs/passkey_qwen3b.yaml", gpu=0); b = run_cfg("configs/ablation_qwen3b.yaml", gpu=1)
    wait([a[0], b[0]], [a[1], b[1]])
elif STAGE == "window_profile":
    a = run_cfg("configs/ablation_window16_qwen3b.yaml", gpu=0); b = run_cfg("configs/ablation_window64_qwen3b.yaml", gpu=1)
    wait([a[0], b[0]], [a[1], b[1]])
    p = sh([sys.executable, "scripts/profile_memory.py", "--model", "Qwen/Qwen2.5-3B-Instruct",
            "--lengths", "4096", "8192", "16384", "--out", "results/profile_qwen3b.jsonl"], 0, "log_profile.txt")
    wait([p], ["log_profile.txt"])
elif STAGE == "llama":
    calibrate("meta-llama/Llama-3.2-3B-Instruct", "profiles/llama3.2-3b-instruct.json")
    main_split("configs/llama3b_generalisation.yaml", ["0.2"], "llama3b_generalisation")
elif STAGE != "figures":
    raise ValueError(f"unknown STAGE {STAGE}")
print(f"stage {STAGE} finished in {(time.time() - t0) / 3600:.2f} h")

In [ ]:
subprocess.run([sys.executable, "scripts/make_figures.py", "--results", "results/*.jsonl", "--out", "figures"])
subprocess.run(["zip", "-qr", "/kaggle/working/results.zip", "results", "profiles", "figures"])
for f in sorted(glob.glob("results/*.jsonl")):
    print(f, sum(1 for _ in open(f)), "rows")
print("Download results.zip from the Output tab.")